# 🌱 Predictor de Fechas Óptimas de Siembra — Estado de Morelos 2026

**Versión especializada para los 36 municipios de Morelos**

| Fuente | Datos usados |
|---|---|
| 🌦️ [CONAGUA / SMN](https://smn.conagua.gob.mx/es/climatologia/informacion-climatologica/normales-climatologicas-por-estado?estado=mor) | Normales Climatológicas 1981-2010 por zona agroecológica |
| 🌿 [SADER](https://www.gob.mx/sader) | Parámetros agronómicos por cultivo |
| 🔬 [INIFAP](https://www.inifap.gob.mx) | Guías técnicas para la región centro-sur |
| 📊 [SIAP 2023-2024](https://nube.agricultura.gob.mx/cierre_agricola/) | Anuario estadístico de producción agrícola Morelos |
| 🗺️ [INEGI](https://www.inegi.org.mx) | Prontuario geográfico municipal |

## Zonas agroecológicas incluidas
- 🏔️ **Zona 1 — Norte Alto** (>2000 msnm): Huitzilac, Tetela del Volcán, Tlalnepantla, Hueyapan, Ocuituco
- 🌄 **Zona 2 — Valles Altos** (1200-2000 msnm): Cuernavaca, Tepoztlán, Jiutepec, Temixco, Emiliano Zapata...
- 🌾 **Zona 3 — Cuenca Oriente** (900-1500 msnm): Cuautla, Yautepec, Ayala, Yecapixtla, Jonacatepec...
- 🌴 **Zona 4 — Sur Cálido** (700-1200 msnm): Zacatepec ⭐, Jojutla, Puente de Ixtla, Tlaquiltenango...

## Instrucciones
1. Ejecuta la celda de abajo (▶ o **Shift+Enter**)
2. Elige tu municipio del menú numerado
3. Selecciona si quieres analizar **un cultivo** o **todos los cultivos**
4. El programa muestra el ranking y guarda un reporte `.txt` y `.json`


In [ ]:
"""
PREDICTOR DE SIEMBRA — ESTADO DE MORELOS 2026
==============================================
Versión especializada para los 36 municipios de Morelos.

Fuentes oficiales:
- CONAGUA / SMN: Normales Climatológicas 1981-2010 por estación
  https://smn.conagua.gob.mx/es/climatologia/informacion-climatologica/normales-climatologicas-por-estado?estado=mor
- SADER / INIFAP: Guías técnicas de cultivos para la región centro-sur
- SIAP (2023-2024): Anuario Estadístico de Producción Agrícola — Morelos
  https://nube.agricultura.gob.mx/cierre_agricola/
- INEGI: Prontuario de Información Geográfica Municipal de Morelos

Zonas agroecológicas de Morelos (INIFAP):
  ZONA 1 — NORTE ALTO  : Huitzilac, Tlalnepantla, Tetela del Volcán, Hueyapan
                          (>2000 msnm, templado-frío, riesgo de heladas)
  ZONA 2 — VALLES ALTOS: Cuernavaca, Tepoztlán, Atlatlahucan, Yecapixtla,
                          Ocuituco, Totolapan, Jiutepec, Temixco, Xochitepec
                          (1200-2000 msnm, semicálido subhúmedo)
  ZONA 3 — CUENCA ORIENTE: Cuautla, Ayala, Yautepec, Emiliano Zapata,
                          Jonacatepec, Jantetelco, Zacualpan de Amilpas,
                          Tepalcingo, Axochiapan, Temoac
                          (900-1500 msnm, cálido subhúmedo)
  ZONA 4 — SUR CÁLIDO  : Jojutla, Zacatepec, Tlaltizapán, Puente de Ixtla,
                          Tlaquiltenango, Amacuzac, Tetecala, Mazatepec,
                          Miacatlán, Coatlán del Río, Coatetelco, Xoxocotla
                          (700-1200 msnm, cálido, zona cañera y arrocera)
"""

import json
from datetime import date, timedelta
from typing import Dict, List, Tuple

MESES = ["Enero","Febrero","Marzo","Abril","Mayo","Junio",
         "Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"]

# =============================================================================
# DATOS CLIMÁTICOS POR ZONA AGROECOLÓGICA
# (Derivados de Normales CONAGUA, estaciones en Morelos 1981-2010)
# Temperatura media mensual (°C) | Precipitación mensual (mm) | Humedad rel. (%)
# =============================================================================

ZONAS_CLIMA: Dict[str, Dict] = {
    "ZONA_1_NORTE_ALTO": {
        "descripcion": "Norte alto / Volcánico",
        "clima_tipo": "Templado subhúmedo C(w)",
        "altitud_ref": "2000-3000 msnm",
        # Basado en estación Huitzilac y Tetela del Volcán (CONAGUA)
        "temp_media":     [9.5, 10.8, 13.2, 15.4, 16.0, 15.2, 14.1, 14.0, 13.6, 12.0, 10.2, 9.0],
        "precipitacion":  [8.2, 6.4,  10.8, 22.0, 56.4, 128.6, 175.2, 162.4, 128.8, 58.2, 14.4, 7.6],
        "humedad_rel":    [72, 65, 58, 56, 60, 74, 82, 83, 82, 76, 70, 72],
        "heladas_riesgo": [True, True, True, False, False, False, False, False, False, False, True, True],
    },
    "ZONA_2_VALLES_ALTOS": {
        "descripcion": "Valles altos / Cuernavaca",
        "clima_tipo": "Semicálido subhúmedo A(C)(w)",
        "altitud_ref": "1200-2000 msnm",
        # Basado en estación Cuernavaca (CONAGUA 17-006)
        "temp_media":     [17.2, 18.6, 21.0, 23.2, 24.4, 23.2, 22.0, 21.8, 21.4, 20.2, 18.4, 17.0],
        "precipitacion":  [8.2,  5.6,  9.4,  22.8, 60.4, 158.6, 188.4, 174.2, 128.6, 52.8, 12.4, 6.8],
        "humedad_rel":    [60, 54, 48, 46, 52, 68, 76, 77, 76, 69, 61, 60],
        "heladas_riesgo": [False]*12,
    },
    "ZONA_3_CUENCA_ORIENTE": {
        "descripcion": "Cuenca oriente / Cuautla-Yautepec",
        "clima_tipo": "Cálido subhúmedo Aw",
        "altitud_ref": "900-1500 msnm",
        # Basado en estación Cuautla y Yautepec (CONAGUA)
        "temp_media":     [18.4, 19.8, 22.4, 25.0, 26.8, 25.6, 24.2, 24.0, 23.4, 21.8, 19.4, 17.8],
        "precipitacion":  [9.4,  6.8,  8.2,  20.4, 58.2, 152.4, 196.8, 186.4, 148.6, 62.4, 14.2, 8.6],
        "humedad_rel":    [57, 51, 46, 44, 50, 67, 75, 76, 74, 67, 58, 57],
        "heladas_riesgo": [False]*12,
    },
    "ZONA_4_SUR_CALIDO": {
        "descripcion": "Sur cálido / Jojutla-Zacatepec",
        "clima_tipo": "Cálido subhúmedo Aw (más cálido)",
        "altitud_ref": "700-1200 msnm",
        # Basado en estación Jojutla y Zacatepec (CONAGUA 17-023, 17-046)
        "temp_media":     [20.8, 22.2, 25.0, 27.8, 29.6, 28.4, 27.0, 26.8, 26.2, 24.4, 21.8, 20.2],
        "precipitacion":  [10.4, 7.2,  7.6,  18.6, 52.4, 164.8, 218.6, 204.2, 162.4, 68.8, 16.4, 9.8],
        "humedad_rel":    [62, 56, 50, 48, 54, 72, 80, 81, 80, 73, 64, 62],
        "heladas_riesgo": [False]*12,
    },
}

# =============================================================================
# 36 MUNICIPIOS DE MORELOS — con zona climática, altitud y cultivos principales
# Fuente: INEGI, SIAP 2023, INIFAP
# =============================================================================

MUNICIPIOS: Dict[str, Dict] = {
    # ── ZONA 1: NORTE ALTO ──────────────────────────────────────────────────
    "Huitzilac": {
        "zona": "ZONA_1_NORTE_ALTO", "altitud": 2650,
        "region": "Norte volcánico",
        "cultivos_principales": ["Papa", "Maíz", "Avena forrajera", "Haba"],
        "nota_local": "Zona de bosque templado. Suelos volcánicos fértiles. Riesgo alto de heladas nov-mar.",
    },
    "Tlalnepantla": {
        "zona": "ZONA_1_NORTE_ALTO", "altitud": 2400,
        "region": "Norte volcánico",
        "cultivos_principales": ["Nopal", "Maíz", "Papa", "Maguey"],
        "nota_local": "Principal productor de nopal del estado. Primer lugar nacional junto con Tepoztlán.",
    },
    "Tetela del Volcán": {
        "zona": "ZONA_1_NORTE_ALTO", "altitud": 2200,
        "region": "Norte volcánico",
        "cultivos_principales": ["Papa", "Maíz", "Durazno", "Pera"],
        "nota_local": "Fruticultura de clima templado. Microclima muy variable por altitud.",
    },
    "Hueyapan": {
        "zona": "ZONA_1_NORTE_ALTO", "altitud": 2100,
        "region": "Norte volcánico",
        "cultivos_principales": ["Maíz", "Frijol", "Papa", "Durazno"],
        "nota_local": "Municipio indígena nahuatl. Agricultura de subsistencia y milpa tradicional.",
    },
    "Ocuituco": {
        "zona": "ZONA_1_NORTE_ALTO", "altitud": 1950,
        "region": "Norte volcánico",
        "cultivos_principales": ["Maíz", "Frijol", "Papa", "Aguacate"],
        "nota_local": "Transición entre zona alta y valles medios.",
    },
    # ── ZONA 2: VALLES ALTOS ─────────────────────────────────────────────────
    "Cuernavaca": {
        "zona": "ZONA_2_VALLES_ALTOS", "altitud": 1510,
        "region": "Valle central",
        "cultivos_principales": ["Flores ornamentales", "Maíz", "Aguacate", "Jitomate"],
        "nota_local": "Capital del estado. Flores ornamentales son el 2° cultivo en valor. +3000 ha de ornamentales.",
    },
    "Jiutepec": {
        "zona": "ZONA_2_VALLES_ALTOS", "altitud": 1300,
        "region": "Valle central",
        "cultivos_principales": ["Flores ornamentales", "Maíz", "Aguacate"],
        "nota_local": "Zona metropolitana de Cuernavaca. Importante en floricultura.",
    },
    "Temixco": {
        "zona": "ZONA_2_VALLES_ALTOS", "altitud": 1260,
        "region": "Valle central",
        "cultivos_principales": ["Maíz", "Flores ornamentales", "Aguacate", "Chile"],
        "nota_local": "Municipio periurbano con vocación agrícola en sus zonas rurales.",
    },
    "Xochitepec": {
        "zona": "ZONA_2_VALLES_ALTOS", "altitud": 1200,
        "region": "Valle central",
        "cultivos_principales": ["Maíz", "Flores", "Caña de azúcar", "Sorgo"],
        "nota_local": "Zona de transición hacia el sur cálido.",
    },
    "Emiliano Zapata": {
        "zona": "ZONA_2_VALLES_ALTOS", "altitud": 1280,
        "region": "Valle central",
        "cultivos_principales": ["Flores ornamentales", "Maíz", "Jitomate"],
        "nota_local": "Tercer municipio más poblado. Gran desarrollo florícola.",
    },
    "Tepoztlán": {
        "zona": "ZONA_2_VALLES_ALTOS", "altitud": 1700,
        "region": "Norte serrano",
        "cultivos_principales": ["Nopal", "Maíz", "Aguacate", "Durazno"],
        "nota_local": "Co-productor del nopal morelense. 3,600 ha de nopal en la región Tepoztlán-Tlalnepantla.",
    },
    "Atlatlahucan": {
        "zona": "ZONA_2_VALLES_ALTOS", "altitud": 1560,
        "region": "Oriente serrano",
        "cultivos_principales": ["Maíz", "Frijol", "Aguacate"],
        "nota_local": "Municipio pequeño de vocación maicera.",
    },
    "Totolapan": {
        "zona": "ZONA_2_VALLES_ALTOS", "altitud": 1650,
        "region": "Oriente serrano",
        "cultivos_principales": ["Nopal", "Maíz", "Aguacate", "Pera"],
        "nota_local": "Nopal y fruticultura de clima templado.",
    },
    # ── ZONA 3: CUENCA ORIENTE ───────────────────────────────────────────────
    "Cuautla": {
        "zona": "ZONA_3_CUENCA_ORIENTE", "altitud": 1310,
        "region": "Valle oriente",
        "cultivos_principales": ["Flores ornamentales", "Maíz", "Sorgo", "Jitomate"],
        "nota_local": "Segunda ciudad del estado. Centro comercial y floricultor del oriente.",
    },
    "Yautepec": {
        "zona": "ZONA_3_CUENCA_ORIENTE", "altitud": 1200,
        "region": "Valle oriente",
        "cultivos_principales": ["Caña de azúcar", "Maíz", "Sorgo", "Flores"],
        "nota_local": "Clima cálido-subhúmedo. Transición caña/maíz según altitud.",
    },
    "Ayala": {
        "zona": "ZONA_3_CUENCA_ORIENTE", "altitud": 1240,
        "region": "Valle oriente",
        "cultivos_principales": ["Maíz", "Sorgo", "Jitomate", "Chile"],
        "nota_local": "Importante productor de granos. 4° municipio más poblado.",
    },
    "Yecapixtla": {
        "zona": "ZONA_3_CUENCA_ORIENTE", "altitud": 1580,
        "region": "Oriente serrano",
        "cultivos_principales": ["Maíz", "Frijol", "Aguacate", "Durazno"],
        "nota_local": "Altitud media con microclima favorable para frutales y maíz.",
    },
    "Jonacatepec": {
        "zona": "ZONA_3_CUENCA_ORIENTE", "altitud": 1180,
        "region": "Suroriente",
        "cultivos_principales": ["Maíz", "Sorgo", "Jitomate", "Pepino"],
        "nota_local": "También llamado Jonacatepec de Leandro Valle.",
    },
    "Jantetelco": {
        "zona": "ZONA_3_CUENCA_ORIENTE", "altitud": 1100,
        "region": "Suroriente",
        "cultivos_principales": ["Maíz", "Sorgo", "Caña de azúcar", "Frijol"],
        "nota_local": "Zona de transición oriente-sur.",
    },
    "Tepalcingo": {
        "zona": "ZONA_3_CUENCA_ORIENTE", "altitud": 980,
        "region": "Suroriente",
        "cultivos_principales": ["Maíz", "Sorgo", "Caña de azúcar", "Jitomate"],
        "nota_local": "Clima más cálido del oriente. Zona cañera incipiente.",
    },
    "Axochiapan": {
        "zona": "ZONA_3_CUENCA_ORIENTE", "altitud": 940,
        "region": "Suroriente",
        "cultivos_principales": ["Maíz", "Sorgo", "Caña de azúcar", "Chile"],
        "nota_local": "Municipio limítrofe con Puebla. Agricultura de temporal.",
    },
    "Zacualpan de Amilpas": {
        "zona": "ZONA_3_CUENCA_ORIENTE", "altitud": 1250,
        "region": "Suroriente",
        "cultivos_principales": ["Maíz", "Frijol", "Aguacate"],
        "nota_local": "Municipio pequeño con vocación maicera.",
    },
    "Temoac": {
        "zona": "ZONA_3_CUENCA_ORIENTE", "altitud": 1380,
        "region": "Oriente serrano",
        "cultivos_principales": ["Maíz", "Frijol", "Aguacate", "Nopal"],
        "nota_local": "Municipio creado en 1977. Agricultura de subsistencia y nopal.",
    },
    # ── ZONA 4: SUR CÁLIDO ───────────────────────────────────────────────────
    "Zacatepec": {
        "zona": "ZONA_4_SUR_CALIDO", "altitud": 917,
        "region": "Sur cañero",
        "cultivos_principales": ["Caña de azúcar", "Sorgo", "Maíz", "Frijol"],
        "nota_local": "Sede del Ingenio Emiliano Zapata. Principal zona cañera del estado. Tu municipio.",
    },
    "Jojutla": {
        "zona": "ZONA_4_SUR_CALIDO", "altitud": 900,
        "region": "Sur cañero",
        "cultivos_principales": ["Caña de azúcar", "Arroz", "Sorgo", "Maíz"],
        "nota_local": "Centro comercial del sur. Zona cañera y arrocera.",
    },
    "Tlaltizapán": {
        "zona": "ZONA_4_SUR_CALIDO", "altitud": 930,
        "region": "Sur cañero",
        "cultivos_principales": ["Caña de azúcar", "Sorgo", "Maíz", "Arroz"],
        "nota_local": "También llamado Tlaltizapán de Zapata.",
    },
    "Puente de Ixtla": {
        "zona": "ZONA_4_SUR_CALIDO", "altitud": 890,
        "region": "Sur cañero",
        "cultivos_principales": ["Arroz", "Caña de azúcar", "Sorgo", "Maíz"],
        "nota_local": "Principal productor de arroz de Morelos. 342 ha de arroz (SIAP 2022).",
    },
    "Tlaquiltenango": {
        "zona": "ZONA_4_SUR_CALIDO", "altitud": 820,
        "region": "Sur cañero",
        "cultivos_principales": ["Caña de azúcar", "Sorgo", "Maíz", "Jitomate"],
        "nota_local": "Municipio más extenso de Morelos (543 km²). Zona baja de la cuenca del Amacuzac.",
    },
    "Amacuzac": {
        "zona": "ZONA_4_SUR_CALIDO", "altitud": 800,
        "region": "Sur cañero",
        "cultivos_principales": ["Caña de azúcar", "Sorgo", "Maíz"],
        "nota_local": "Zona cañera sur. Municipio pequeño.",
    },
    "Tetecala": {
        "zona": "ZONA_4_SUR_CALIDO", "altitud": 880,
        "region": "Poniente cálido",
        "cultivos_principales": ["Caña de azúcar", "Maíz", "Sorgo"],
        "nota_local": "Municipio pequeño con alta vocación cañera.",
    },
    "Mazatepec": {
        "zona": "ZONA_4_SUR_CALIDO", "altitud": 940,
        "region": "Poniente cálido",
        "cultivos_principales": ["Maíz", "Caña de azúcar", "Sorgo", "Aguacate"],
        "nota_local": "Zona poniente-sur. Clima cálido con más variación.",
    },
    "Miacatlán": {
        "zona": "ZONA_4_SUR_CALIDO", "altitud": 980,
        "region": "Poniente cálido",
        "cultivos_principales": ["Maíz", "Sorgo", "Aguacate", "Caña de azúcar"],
        "nota_local": "Zona poniente con microclima más fresco que el sur extremo.",
    },
    "Coatlán del Río": {
        "zona": "ZONA_4_SUR_CALIDO", "altitud": 860,
        "region": "Poniente cálido",
        "cultivos_principales": ["Caña de azúcar", "Maíz", "Sorgo"],
        "nota_local": "Municipio pequeño. Vocación cañera.",
    },
    "Coatetelco": {
        "zona": "ZONA_4_SUR_CALIDO", "altitud": 920,
        "region": "Poniente cálido",
        "cultivos_principales": ["Maíz", "Sorgo", "Caña de azúcar"],
        "nota_local": "Municipio indígena creado en 2017.",
    },
    "Xoxocotla": {
        "zona": "ZONA_4_SUR_CALIDO", "altitud": 900,
        "region": "Sur cañero",
        "cultivos_principales": ["Caña de azúcar", "Maíz", "Sorgo"],
        "nota_local": "Municipio indígena creado en 2017.",
    },
    "Tepozlán (San José de los Laureles)": {
        "zona": "ZONA_2_VALLES_ALTOS", "altitud": 1600,
        "region": "Norte serrano",
        "cultivos_principales": ["Maíz", "Frijol", "Aguacate"],
        "nota_local": "Comunidad dentro de la región de Tepoztlán.",
    },
}

# =============================================================================
# CULTIVOS RELEVANTES PARA MORELOS
# (Parámetros agronómicos SADER/INIFAP, adaptados a condiciones de Morelos)
# =============================================================================

CULTIVOS: Dict[str, Dict] = {
    "Maíz (temporal)": {
        "emoji": "🌽",
        "nombre_cientifico": "Zea mays",
        "temp_min_germ": 10, "temp_optima": 25, "temp_max": 35, "temp_min_crec": 15,
        "precip_ciclo_mm": 500, "ciclo_dias": 120, "dias_a_cosecha": 90,
        "humedad_optima": (60, 80),
        "requiere_lluvia": True,
        "notas": "Ciclo PV (mayo-oct) en temporal. Toda Morelos es apta. "
                 "Maíz criollo con alto valor cultural. INIFAP recomienda variedades VS-536 y H-311 para valles.",
        "fuente": "INIFAP — Guía Técnica Maíz Grano, Región Centro-Sur",
    },
    "Caña de azúcar": {
        "emoji": "🌿",
        "nombre_cientifico": "Saccharum officinarum",
        "temp_min_germ": 18, "temp_optima": 28, "temp_max": 38, "temp_min_crec": 20,
        "precip_ciclo_mm": 1200, "ciclo_dias": 365, "dias_a_cosecha": 330,
        "humedad_optima": (60, 80),
        "requiere_lluvia": True,
        "notas": "Cultivo más importante de Morelos (52.5% producción estatal). "
                 "Ingenios: Emiliano Zapata (Zacatepec) y Plan de Ayala (Ayala). "
                 "Zafra: nov-mayo. Siembra de plantones o esquejes.",
        "fuente": "SADER — Norma Oficial Mexicana Caña de Azúcar; SIAP Cierre 2023",
    },
    "Arroz": {
        "emoji": "🌾",
        "nombre_cientifico": "Oryza sativa",
        "temp_min_germ": 18, "temp_optima": 28, "temp_max": 38, "temp_min_crec": 20,
        "precip_ciclo_mm": 900, "ciclo_dias": 130, "dias_a_cosecha": 110,
        "humedad_optima": (75, 90),
        "requiere_lluvia": True,
        "notas": "Morelos es productor histórico de arroz. "
                 "Zona sur (Puente de Ixtla, Jojutla). Requiere riego y altas temperaturas. "
                 "Variedad Milagro Filipino muy difundida.",
        "fuente": "INIFAP — Tecnología para Producción de Arroz en el Sur de México",
    },
    "Sorgo": {
        "emoji": "🌾",
        "nombre_cientifico": "Sorghum bicolor",
        "temp_min_germ": 15, "temp_optima": 27, "temp_max": 38, "temp_min_crec": 18,
        "precip_ciclo_mm": 400, "ciclo_dias": 120, "dias_a_cosecha": 90,
        "humedad_optima": (50, 70),
        "requiere_lluvia": True,
        "notas": "Segundo grano más sembrado en Morelos. Alta tolerancia a sequía. "
                 "Ciclo PV (jun-oct). Ideal en zona sur y oriente.",
        "fuente": "SAGARPA — Tecnología de Producción de Sorgo Grano",
    },
    "Jitomate / Tomate rojo": {
        "emoji": "🍅",
        "nombre_cientifico": "Solanum lycopersicum",
        "temp_min_germ": 15, "temp_optima": 24, "temp_max": 32, "temp_min_crec": 16,
        "precip_ciclo_mm": 300, "ciclo_dias": 120, "dias_a_cosecha": 80,
        "humedad_optima": (60, 70),
        "requiere_lluvia": False,
        "notas": "4° cultivo por valor en Morelos (SIAP 2023). "
                 "Prefiere riego controlado. Ciclos OI (oct-mar) y PV (abr-ago). "
                 "Alta humedad favorece tizón tardío.",
        "fuente": "INIFAP — Tecnología de Producción de Tomate en el Centro de México",
    },
    "Nopal (verdura)": {
        "emoji": "🌵",
        "nombre_cientifico": "Opuntia ficus-indica",
        "temp_min_germ": 12, "temp_optima": 22, "temp_max": 32, "temp_min_crec": 15,
        "precip_ciclo_mm": 350, "ciclo_dias": 90, "dias_a_cosecha": 45,
        "humedad_optima": (40, 65),
        "requiere_lluvia": False,
        "notas": "Morelos es 1° lugar nacional en nopal/nopalitos. "
                 "3,600 ha en Tlalnepantla, Tlayacapan, Totolapan, Tepoztlán. "
                 "Producción perenne — siembra de pencas para establecimiento.",
        "fuente": "SIAP 2024 — Anuario Estadístico; INIFAP Morelos",
    },
    "Flores ornamentales": {
        "emoji": "🌸",
        "nombre_cientifico": "Varias especies",
        "temp_min_germ": 14, "temp_optima": 20, "temp_max": 28, "temp_min_crec": 15,
        "precip_ciclo_mm": 600, "ciclo_dias": 180, "dias_a_cosecha": 90,
        "humedad_optima": (60, 75),
        "requiere_lluvia": False,
        "notas": "3,000 ha en Morelos. 2° cultivo en valor tras la caña. "
                 "Municipios: Cuautla, Yautepec, Cuernavaca, Emiliano Zapata, Jiutepec. "
                 "Rosas, nochebuenas, crisantemos, gerberas.",
        "fuente": "SAGARPA — Programa de Floricultura; SIAP Cierre 2023",
    },
    "Aguacate": {
        "emoji": "🥑",
        "nombre_cientifico": "Persea americana",
        "temp_min_germ": 16, "temp_optima": 22, "temp_max": 30, "temp_min_crec": 14,
        "precip_ciclo_mm": 1200, "ciclo_dias": 365, "dias_a_cosecha": 270,
        "humedad_optima": (65, 80),
        "requiere_lluvia": True,
        "notas": "Morelos ocupa el 5° lugar nacional en aguacate (SIAP 2024). "
                 "Zonas altas y medias: Tepoztlán, Yecapixtla, Atlatlahucan, Mazatepec. "
                 "Siembra de plantones injertados en primavera.",
        "fuente": "SAGARPA — Tecnología para Producción de Aguacate; SIAP 2024",
    },
    "Frijol": {
        "emoji": "🫘",
        "nombre_cientifico": "Phaseolus vulgaris",
        "temp_min_germ": 12, "temp_optima": 22, "temp_max": 32, "temp_min_crec": 14,
        "precip_ciclo_mm": 350, "ciclo_dias": 90, "dias_a_cosecha": 75,
        "humedad_optima": (50, 70),
        "requiere_lluvia": True,
        "notas": "Cultivo de milpa, asociado con maíz. "
                 "Siembra en junio-julio en temporal. Variedades locales negro y bayo.",
        "fuente": "INIFAP — Guía Técnica Frijol Región Centro-Sur",
    },
    "Papa": {
        "emoji": "🥔",
        "nombre_cientifico": "Solanum tuberosum",
        "temp_min_germ": 7, "temp_optima": 18, "temp_max": 25, "temp_min_crec": 10,
        "precip_ciclo_mm": 500, "ciclo_dias": 110, "dias_a_cosecha": 90,
        "humedad_optima": (70, 85),
        "requiere_lluvia": False,
        "notas": "Exclusiva de zona norte alta (Huitzilac, Tetela, Ocuituco). "
                 "Ciclo mar-jun o ago-nov. Suelos volcánicos ideales.",
        "fuente": "INIFAP — Guía Técnica Papa para el Centro de México",
    },
    "Chile / Pimiento": {
        "emoji": "🌶️",
        "nombre_cientifico": "Capsicum annuum",
        "temp_min_germ": 16, "temp_optima": 26, "temp_max": 35, "temp_min_crec": 18,
        "precip_ciclo_mm": 400, "ciclo_dias": 150, "dias_a_cosecha": 90,
        "humedad_optima": (60, 75),
        "requiere_lluvia": False,
        "notas": "Zonas media y baja de Morelos. Variedades chile de agua y jalapeño. "
                 "Ciclo OI con riego o PV en temporal.",
        "fuente": "INIFAP — Manual Técnico de Chile para el Trópico",
    },
    "Durazno": {
        "emoji": "🍑",
        "nombre_cientifico": "Prunus persica",
        "temp_min_germ": 5, "temp_optima": 18, "temp_max": 26, "temp_min_crec": 8,
        "precip_ciclo_mm": 600, "ciclo_dias": 365, "dias_a_cosecha": 120,
        "humedad_optima": (55, 75),
        "requiere_lluvia": False,
        "notas": "Morelos ocupa el 10° lugar nacional en durazno (SIAP 2024). "
                 "Solo viable en zona norte alta (>1800 msnm): Tetela, Huitzilac, Ocuituco. "
                 "Requiere horas-frío para florecer.",
        "fuente": "INIFAP — Tecnología para Producción de Durazno en México",
    },
}

# =============================================================================
# MOTOR DE ANÁLISIS
# =============================================================================

def calcular_score_mes(mes_idx: int, cultivo: Dict, clima: Dict) -> Tuple[float, List[str]]:
    score = 100.0
    razones = []

    temp     = clima["temp_media"][mes_idx]
    precip   = clima["precipitacion"][mes_idx]
    humedad  = clima["humedad_rel"][mes_idx]
    helada   = clima["heladas_riesgo"][mes_idx]

    t_min  = cultivo["temp_min_germ"]
    t_opt  = cultivo["temp_optima"]
    t_max  = cultivo["temp_max"]

    # --- Temperatura ---
    if temp < t_min:
        pen = min(65, (t_min - temp) * 9)
        score -= pen
        razones.append(f"⚠️  Temp {temp:.1f}°C bajo mínimo de germinación ({t_min}°C)")
    elif temp > t_max:
        pen = min(55, (temp - t_max) * 7)
        score -= pen
        razones.append(f"🔥 Temp {temp:.1f}°C excede máximo tolerable ({t_max}°C)")
    else:
        rango = t_max - t_min
        dist = abs(temp - t_opt) / (rango / 2 + 0.1)
        bonus = max(0, 12 * (1 - dist))
        score += bonus
        razones.append(f"✅ Temperatura {temp:.1f}°C — rango óptimo ({t_min}-{t_max}°C)")

    # --- Heladas ---
    if helada:
        score -= 45
        razones.append("❄️  Riesgo de heladas — crítico para la mayoría de cultivos")

    # --- Precipitación ---
    precip_mensual = cultivo["precip_ciclo_mm"] / (cultivo["ciclo_dias"] / 30)
    if cultivo["requiere_lluvia"]:
        if precip < precip_mensual * 0.3:
            score -= 22
            razones.append(f"🏜️  Lluvia insuficiente ({precip:.0f}mm) — requerida {precip_mensual:.0f}mm/mes")
        elif precip >= precip_mensual * 0.75:
            score += 8
            razones.append(f"🌧️  Precipitación favorable ({precip:.0f}mm)")
        else:
            razones.append(f"💧 Lluvia moderada ({precip:.0f}mm) — puede requerir riego de apoyo")
    else:
        if precip > 220:
            score -= 12
            razones.append(f"🌊 Precipitación excesiva ({precip:.0f}mm) — riesgo de enfermedades fungosas")

    # --- Humedad ---
    h_min, h_max = cultivo["humedad_optima"]
    if h_min <= humedad <= h_max:
        score += 5
        razones.append(f"💧 Humedad relativa óptima ({humedad}%)")
    elif humedad > h_max + 12:
        score -= 8
        razones.append(f"💦 Alta humedad ({humedad}%) — mayor riesgo de hongos")

    return max(0.0, min(100.0, score)), razones


def predecir(municipio: str, cultivo_nombre: str, año: int = 2026) -> Dict:
    mun  = MUNICIPIOS[municipio]
    zona = ZONAS_CLIMA[mun["zona"]]
    cult = CULTIVOS[cultivo_nombre]

    resultados = []
    for i in range(12):
        score, razones = calcular_score_mes(i, cult, zona)
        f_inicio  = date(año, i + 1, 15)
        f_cosecha = f_inicio + timedelta(days=cult["dias_a_cosecha"])
        resultados.append({
            "mes": MESES[i], "mes_idx": i,
            "score": round(score, 1),
            "fecha_siembra": f_inicio.strftime("%d/%b/%Y"),
            "fecha_cosecha": f_cosecha.strftime("%d/%b/%Y"),
            "temp": zona["temp_media"][i],
            "precipitacion": zona["precipitacion"][i],
            "humedad": zona["humedad_rel"][i],
            "helada": zona["heladas_riesgo"][i],
            "razones": razones,
        })

    ranking = sorted(resultados, key=lambda x: x["score"], reverse=True)
    return {
        "municipio": municipio,
        "zona_desc": zona["descripcion"],
        "zona_clima": zona["clima_tipo"],
        "altitud": mun["altitud"],
        "region": mun["region"],
        "cultivos_tipicos": mun["cultivos_principales"],
        "nota_local": mun["nota_local"],
        "cultivo": cultivo_nombre,
        "emoji": cult["emoji"],
        "ciclo_dias": cult["ciclo_dias"],
        "nota_cultivo": cult["notas"],
        "fuente_cultivo": cult["fuente"],
        "año": año,
        "mejor": ranking[0],
        "top3": ranking[:3],
        "todos": resultados,
        "buenos": [r for r in ranking if r["score"] >= 70],
        "regulares": [r for r in ranking if 40 <= r["score"] < 70],
        "malos": [r for r in ranking if r["score"] < 40],
    }


def barra(score: float, largo: int = 22) -> str:
    llenos = int((score / 100) * largo)
    c = "█" if score >= 70 else ("▒" if score >= 40 else "░")
    return c * llenos + "·" * (largo - llenos)


def semaforo(score: float) -> str:
    if score >= 80: return "🟢 EXCELENTE"
    if score >= 65: return "🟡 BUENO    "
    if score >= 40: return "🟠 REGULAR  "
    return              "🔴 NO APTO  "


def reporte(r: Dict) -> str:
    sep  = "=" * 72
    sep2 = "-" * 72
    L = []
    L.append(sep)
    L.append(f"  🌱 PREDICTOR DE SIEMBRA — ESTADO DE MORELOS {r['año']}")
    L.append(f"  Datos: CONAGUA · SADER · INIFAP · SIAP")
    L.append(sep)
    L.append(f"  📍 Municipio  : {r['municipio']} — {r['region']}")
    L.append(f"  🗺️  Zona       : {r['zona_desc']} ({r['zona_clima']})")
    L.append(f"  ⛰️  Altitud    : {r['altitud']} msnm")
    L.append(f"  🌿 Cultivos típicos: {', '.join(r['cultivos_tipicos'])}")
    L.append(f"  📝 {r['nota_local']}")
    L.append("")
    L.append(f"  {r['emoji']} Cultivo a sembrar: {r['cultivo']}")
    L.append(f"  ⏱️  Ciclo     : ~{r['ciclo_dias']} días")
    L.append("")
    L.append(sep2)
    L.append("  🏆 TOP 3 MEJORES MESES PARA SEMBRAR EN 2026")
    L.append(sep2)
    for pos, m in enumerate(r["top3"], 1):
        L.append(f"  #{pos}  {m['mes']:<12} [{barra(m['score'])}] {m['score']:>5.0f}/100  {semaforo(m['score'])}")
        L.append(f"       Siembra: {m['fecha_siembra']}  →  Cosecha aprox: {m['fecha_cosecha']}")
        L.append(f"       Temp: {m['temp']:.1f}°C  |  Lluvia: {m['precipitacion']:.0f}mm  |  Humedad: {m['humedad']}%")
        for raz in m["razones"][:3]:
            L.append(f"       {raz}")
        L.append("")

    L.append(sep2)
    L.append("  📊 RANKING COMPLETO — 12 MESES")
    L.append(sep2)
    L.append(f"  {'Mes':<13}  {'Calidad':15} {'Score':>6}  {'Temp':>6}  {'Lluvia':>7}  {'Helada':>6}  Siembra→Cosecha")
    L.append(f"  {'-'*13}  {'-'*15} {'-'*6}  {'-'*6}  {'-'*7}  {'-'*6}  {'-'*22}")
    for m in r["todos"]:
        h = "❄️ SÍ" if m["helada"] else "  NO "
        L.append(
            f"  {m['mes']:<13}  {semaforo(m['score'])}  {m['score']:>5.0f}%"
            f"  {m['temp']:>5.1f}°C  {m['precipitacion']:>5.0f}mm  {h}"
            f"  {m['fecha_siembra']}→{m['fecha_cosecha']}"
        )

    L.append("")
    L.append(sep2)
    L.append("  📋 NOTAS DEL CULTIVO (SADER/INIFAP)")
    L.append(sep2)
    # Wrap text at 68 chars
    texto = r["nota_cultivo"]
    while len(texto) > 68:
        corte = texto[:68].rfind(" ")
        L.append(f"  {texto[:corte]}")
        texto = texto[corte+1:]
    L.append(f"  {texto}")
    L.append(f"  Fuente: {r['fuente_cultivo']}")
    L.append("")
    L.append("  ⚠️  AVISO: Prototipo con Normales Climatológicas históricas (1981-2010).")
    L.append("  Para decisiones reales consultar al CADER o técnico INIFAP de tu región.")
    L.append("  CADER Morelos: https://www.gob.mx/sader  |  INIFAP: https://www.inifap.gob.mx")
    L.append(sep)
    return "\n".join(L)


def comparar_cultivos(municipio: str, año: int = 2026):
    mun = MUNICIPIOS[municipio]
    print(f"\n{'='*72}")
    print(f"  📊 TODOS LOS CULTIVOS — {municipio.upper()} ({mun['region']}, {mun['altitud']} msnm)")
    print(f"  Zona: {ZONAS_CLIMA[mun['zona']]['descripcion']} — {ZONAS_CLIMA[mun['zona']]['clima_tipo']}")
    print(f"{'='*72}\n")
    print(f"  {'Cultivo':<25} {'Mejor mes':<12} {'Score':>6}  {'Calidad':<14}  Siembra→Cosecha")
    print(f"  {'-'*25} {'-'*12} {'-'*6}  {'-'*14}  {'-'*22}")
    for nombre in CULTIVOS:
        r = predecir(municipio, nombre, año)
        m = r["mejor"]
        e = CULTIVOS[nombre]["emoji"]
        print(f"  {e} {nombre:<23} {m['mes']:<12} {m['score']:>5.0f}%  {semaforo(m['score'])}  {m['fecha_siembra']}→{m['fecha_cosecha']}")
    print()


def menu():
    print("\n" + "=" * 72)
    print("  🌱 PREDICTOR DE FECHAS ÓPTIMAS DE SIEMBRA")
    print("  ESTADO DE MORELOS — 2026")
    print("  Fuentes: CONAGUA | SADER | INIFAP | SIAP")
    print("=" * 72)

    # ── Selección de municipio ───────────────────────────────────────────────
    municipios = list(MUNICIPIOS.keys())
    print(f"\n📍 Selecciona tu municipio ({len(municipios)} en total):\n")

    # Agrupar por zona
    grupos = {
        "🏔️  ZONA 1 — Norte Alto (>2000 msnm, con riesgo de heladas)": "ZONA_1_NORTE_ALTO",
        "🌄  ZONA 2 — Valles Altos (1200-2000 msnm, semicálido)":       "ZONA_2_VALLES_ALTOS",
        "🌾  ZONA 3 — Cuenca Oriente (900-1500 msnm, cálido)":          "ZONA_3_CUENCA_ORIENTE",
        "🌴  ZONA 4 — Sur Cálido (700-1200 msnm, cañero/arrocero)":     "ZONA_4_SUR_CALIDO",
    }
    indices = {}
    idx = 1
    for titulo, zona_key in grupos.items():
        print(f"  {titulo}")
        for nombre, datos in MUNICIPIOS.items():
            if datos["zona"] == zona_key:
                marca = " ◀ (tu municipio)" if nombre == "Zacatepec" else ""
                print(f"    [{idx:2d}] {nombre:<30} {datos['altitud']:>5} msnm{marca}")
                indices[idx] = nombre
                idx += 1
        print()

    while True:
        try:
            sel = int(input("  Ingresa el número de tu municipio: "))
            if sel in indices:
                municipio = indices[sel]
                break
            print("  Número fuera de rango.")
        except ValueError:
            print("  Ingresa un número válido.")

    print(f"\n  ✅ Municipio seleccionado: {municipio}")
    mun = MUNICIPIOS[municipio]
    print(f"     Zona: {ZONAS_CLIMA[mun['zona']]['descripcion']} | {mun['altitud']} msnm")
    print(f"     Cultivos típicos: {', '.join(mun['cultivos_principales'])}")

    # ── Modo de análisis ─────────────────────────────────────────────────────
    print(f"\n  ¿Qué deseas hacer?\n")
    print("    [1] Analizar un cultivo específico")
    print("    [2] Comparar TODOS los cultivos para este municipio")

    while True:
        try:
            modo = int(input("\n  Elige una opción: "))
            if modo in [1, 2]: break
        except ValueError: pass

    if modo == 2:
        comparar_cultivos(municipio)
        return

    # ── Selección de cultivo ─────────────────────────────────────────────────
    cultivos = list(CULTIVOS.keys())
    print(f"\n🌿 Cultivos disponibles:\n")
    for i, c in enumerate(cultivos, 1):
        d = CULTIVOS[c]
        tipico = " ⭐ (típico de tu zona)" if c.split(" ")[0] in " ".join(mun["cultivos_principales"]) else ""
        print(f"  [{i:2d}] {d['emoji']} {c:<30} ciclo ~{d['ciclo_dias']} días{tipico}")

    while True:
        try:
            sel2 = int(input("\n  Número del cultivo: "))
            if 1 <= sel2 <= len(cultivos):
                cultivo = cultivos[sel2 - 1]
                break
            print("  Número fuera de rango.")
        except ValueError:
            print("  Ingresa un número válido.")

    print(f"\n⏳ Analizando {cultivo} en {municipio}...")
    r = predecir(municipio, cultivo, 2026)
    print("\n" + reporte(r))

    # Guardar
    nombre = f"siembra_{municipio.lower().replace(' ', '_').replace('/', '_')}_{cultivo.lower().replace(' ', '_').replace('/', '_')}_2026.txt"
    for ch in "áéíóúüñ": nombre = nombre.replace(ch, "")
    with open(nombre, "w", encoding="utf-8") as f:
        f.write(reporte(r))
    print(f"\n  💾 Reporte guardado: {nombre}")

    j = nombre.replace(".txt", ".json")
    with open(j, "w", encoding="utf-8") as f:
        json.dump(r, f, ensure_ascii=False, indent=2, default=str)
    print(f"  💾 Datos JSON:       {j}")


# =============================================================================
def _es_jupyter():
    try:
        shell = get_ipython().__class__.__name__  # type: ignore
        return shell in ("ZMQInteractiveShell", "TerminalInteractiveShell")
    except NameError:
        return False


if __name__ == "__main__":
    import sys
    if _es_jupyter():
        menu()
    elif len(sys.argv) == 3 and not sys.argv[1].startswith("-"):
        mun_arg, cult_arg = sys.argv[1], sys.argv[2]
        if mun_arg not in MUNICIPIOS:
            print(f"Municipio no encontrado. Disponibles: {list(MUNICIPIOS.keys())}")
            sys.exit(1)
        if cult_arg not in CULTIVOS:
            print(f"Cultivo no encontrado. Disponibles: {list(CULTIVOS.keys())}")
            sys.exit(1)
        r = predecir(mun_arg, cult_arg)
        print(reporte(r))
    elif len(sys.argv) == 2 and sys.argv[1] == "--comparar":
        mun_comp = input("Municipio: ")
        if mun_comp in MUNICIPIOS:
            comparar_cultivos(mun_comp)
        else:
            print("Municipio no encontrado.")
    else:
        menu()
